## バージョン情報
### V0.1

```
id : id
health_condition : 健康状態  (y)

==========================================

1  sleep_duration : 睡眠時間
2  heart_rate : 心拍数
3  bmi : BMI
4  calorie_expenditure :  カロリー消費量
5  step_count : ステップ数
6  exercise_duration : 運動時間
7  water_intake : 水分摂取量
8  diet_type : 食事タイプ
9  stress_level : ストレスレベル
10 sleep_quality : 睡眠の質
11 physical_activity_level : 身体活動レベル
12 smoking_alcohol : 喫煙・飲酒
13 gender : 性別

==========================================
追加

14 short_sleep_flag : 睡眠時間が6時間未満なら「1」、そうでない（空白も含む）なら「0」


```



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

In [2]:
def df_Data_Cleansing(df):
    cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

            #14 short_sleep_flag
            df['short_sleep_flag'] = (df['sleep_duration'] < 6.0).astype(int)

            
    return df

In [3]:
#読み込み
df = df_Data_Cleansing(pd.read_csv('data/train.csv',encoding="utf-8")) 
df_test = df_Data_Cleansing(pd.read_csv('data/test.csv',encoding="utf-8"))

In [4]:
df

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,short_sleep_flag
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female,1
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other,1
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male,1
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female,1
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,690083,at-risk,6.31,69.7,24.11,2157.0,NaN,30.8,3.00,non-veg,high,poor,active,yes,female,0
690084,690084,at-risk,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46,veg,medium,average,moderate,no,male,1
690085,690085,fit,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57,non-veg,NaN,average,active,no,male,0
690086,690086,at-risk,6.74,73.0,18.73,2061.0,13366.0,56.6,2.60,balanced,NaN,average,active,yes,male,0


In [5]:
df_test

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,short_sleep_flag
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male,1
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other,0
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN,0
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other,0
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295748,985836,7.13,77.5,22.20,2221.0,13981.0,65.6,2.17,balanced,low,average,active,occasional,other,0
295749,985837,6.25,69.5,20.93,2242.0,2370.0,23.6,1.98,balanced,NaN,average,moderate,yes,female,0
295750,985838,NaN,77.0,23.62,2111.0,7647.0,25.1,2.18,veg,high,average,sedentary,no,other,0
295751,985839,7.25,66.3,21.47,2009.0,4672.0,25.1,2.02,veg,high,poor,moderate,no,female,0


In [6]:
#特徴量(X)とターゲット(y)の分割
X = df.drop(['id', 'health_condition'], axis=1)
y = df['health_condition']

#ターゲット(y)を数値に変換
le = LabelEncoder()
y_encoded = le.fit_transform(y)

#学習
model = lgb.LGBMClassifier(class_weight='balanced', random_state=42)
model.fit(X, y_encoded)

#予測と提出ファイルの作成
X_test = df_test.drop(['id'], axis=1)
preds = model.predict(X_test)

#予測結果（数値）を元のテキストラベルに戻す
submission_labels = le.inverse_transform(preds)

submission = pd.DataFrame({
    'id': df_test['id'],
    'health_condition': submission_labels
})
submission.to_csv('submission.csv', index=False)
X_test.head(100).to_csv("submit_x_x_x_x_x_x.csv", index=False)

print("""=========================



提出用ファイル 'submission.csv' を作成しました☆

""")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004889 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1810
[LightGBM] [Info] Number of data points in the train set: 690088, number of used features: 14
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612



提出用ファイル 'submission.csv' を作成しました☆




In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# 学習データを8:2に分割
X_train, X_valid, y_train, y_valid = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# モデルの学習
model = lgb.LGBMClassifier(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# 手元のデータで予測してスコアを計算
valid_preds = model.predict(X_valid)
score = balanced_accuracy_score(y_valid, valid_preds)

print(f"""=========================

学習用Balanced Accuracy: {score:.5f}

""")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003417 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1810
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 14
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612

学習用Balanced Accuracy: 0.95007


